# Занятие 8. Семинар: деревья решений, бэггинг и случайный лес

План занятия:

1. дерево как набор правил «если — то»;
2. как дерево выбирает вопрос: индекс Джини;
3. глубина дерева и переобучение, параметры регуляризации;
4. важность признаков;
5. пропущенные значения;
6. дерево для регрессии, дерево как линейная модель;
7. неустойчивость дерева и разложение ошибки на смещение и разброс;
8. бутстрэп, бэггинг и оценка вне выборки;
9. метод случайных подпространств, случайный лес, границы kNN и деревьев;
10. изолирующий лес для поиска аномалий.

Выводы формул, отсечение по сложности и свойства критериев — в конспекте
`notes_08_trees.ipynb`.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from sklearn.datasets import load_wine, make_circles, make_classification, make_moons
from sklearn.ensemble import BaggingClassifier, BaggingRegressor, IsolationForest, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
CLASS_COLORS = ["#0072CE", "#C98A3C", "#4DAF4A"]
SEED = 42
FEATURES_2D = ["flavanoids", "color_intensity"]
LOWER_IS_BETTER = ("листьев",)


def show_table(frame, digits=3, axis=0, lower_is_better=False):
    """Таблица с заливкой: лучшее значение ярче; для числа листьев лучше меньшее."""
    numeric = list(frame.select_dtypes("number").columns)
    lower = [c for c in numeric if lower_is_better or any(key in str(c) for key in LOWER_IS_BETTER)]
    higher = [c for c in numeric if c not in lower]

    styled = frame.style.format(precision=digits, subset=numeric)
    if higher:
        styled = styled.background_gradient(cmap="Greens", subset=higher, axis=axis)
    if lower:
        styled = styled.background_gradient(cmap="Greens_r", subset=lower, axis=axis)
    return styled


SUBSCRIPT = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")


def leaf_regions(model, features):
    """Листья дерева слева направо: номер узла и границы области (low, high) по каждому признаку."""
    structure = model.tree_
    leaves = []

    def walk(node, bounds):
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            leaves.append((node, bounds))
            return

        name = features[structure.feature[node]]
        threshold = structure.threshold[node]
        low, high = bounds[name]

        left_bounds = {**bounds, name: (low, min(high, threshold))}     # ответ «да»: x <= threshold
        right_bounds = {**bounds, name: (max(low, threshold), high)}   # ответ «нет»: x > threshold

        walk(left_child, left_bounds)
        walk(right_child, right_bounds)

    unbounded = {name: (-np.inf, np.inf) for name in features}
    walk(0, unbounded)
    return leaves


def rule_text(bounds):
    """Конъюнкция условий пути; условия на один признак объединены в интервал."""
    conditions = []

    for name, (low, high) in bounds.items():
        has_low = np.isfinite(low)
        has_high = np.isfinite(high)

        if has_low and has_high:
            conditions.append(f"[{low:.2f} < {name} ≤ {high:.2f}]")
        elif has_high:
            conditions.append(f"[{name} ≤ {high:.2f}]")
        elif has_low:
            conditions.append(f"[{name} > {low:.2f}]")

    text = " ∧ ".join(conditions)
    return text


def draw_tree_rules(model, X, y, features=FEATURES_2D, class_names=("сорт 0", "сорт 1", "сорт 2")):
    """Области листьев, схема дерева и правила листьев на одном рисунке."""
    structure = model.tree_
    leaves = leaf_regions(model, features)

    leaf_number = {}
    leaf_class = {}
    for number, (node, _) in enumerate(leaves, start=1):
        leaf_number[node] = number
        leaf_class[node] = int(np.argmax(structure.value[node][0]))

    fig = plt.figure(figsize=(17, 10))
    grid = fig.add_gridspec(2, 2, height_ratios=[3, 1.3], width_ratios=[1, 1.2], hspace=0.15, wspace=0.08)
    ax_regions = fig.add_subplot(grid[0, 0])
    ax_tree = fig.add_subplot(grid[0, 1])
    ax_rules = fig.add_subplot(grid[1, :])

    # --- области листьев на плоскости признаков ---
    x_name, y_name = features
    x_lim = (X[x_name].min() - 0.3, X[x_name].max() + 0.3)
    y_lim = (X[y_name].min() - 0.5, X[y_name].max() + 0.5)

    for node, bounds in leaves:
        x_low = max(bounds[x_name][0], x_lim[0])
        x_high = min(bounds[x_name][1], x_lim[1])
        y_low = max(bounds[y_name][0], y_lim[0])
        y_high = min(bounds[y_name][1], y_lim[1])
        color = CLASS_COLORS[leaf_class[node]]

        rectangle = plt.Rectangle((x_low, y_low), x_high - x_low, y_high - y_low,
                                  facecolor=color, alpha=0.22, edgecolor=BLACK, lw=1.2)
        ax_regions.add_patch(rectangle)

        label = f"R{leaf_number[node]}".translate(SUBSCRIPT)
        center = ((x_low + x_high) / 2, (y_low + y_high) / 2)
        ax_regions.text(*center, label, ha="center", va="center", fontsize=16, color=BLACK, zorder=4,
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.7))

    for k, name in enumerate(class_names):
        is_class = y == k
        ax_regions.scatter(X.loc[is_class, x_name], X.loc[is_class, y_name], s=18,
                           color=CLASS_COLORS[k], edgecolor="white", lw=0.4, label=name, zorder=3)

    ax_regions.set_xlim(x_lim)
    ax_regions.set_ylim(y_lim)
    ax_regions.set_xlabel(x_name)
    ax_regions.set_ylabel(y_name)
    ax_regions.grid(False)
    ax_regions.legend(loc="upper right")

    # --- схема дерева: листья по горизонтали слева направо, узел посередине между потомками ---
    positions = {}
    next_leaf_x = iter(range(len(leaves)))

    def place(node, depth):
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            positions[node] = (next(next_leaf_x), -depth)
            return

        place(left_child, depth + 1)
        place(right_child, depth + 1)
        middle_x = (positions[left_child][0] + positions[right_child][0]) / 2
        positions[node] = (middle_x, -depth)

    place(0, 0)

    for node, (x, y_pos) in positions.items():
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            counts = np.round(structure.value[node][0] * structure.n_node_samples[node]).astype(int)
            name = class_names[leaf_class[node]]
            label = f"R{leaf_number[node]}".translate(SUBSCRIPT) + f": {name}\n{counts.tolist()}"
            box = dict(boxstyle="round,pad=0.4", facecolor=CLASS_COLORS[leaf_class[node]], alpha=0.3, edgecolor=BLACK)
        else:
            feature_name = features[structure.feature[node]]
            label = f"{feature_name} ≤ {structure.threshold[node]:.2f}"
            box = dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=GREY)

            for child, answer in [(left_child, "да"), (right_child, "нет")]:
                child_x, child_y = positions[child]
                ax_tree.annotate("", xy=(child_x, child_y + 0.12), xytext=(x, y_pos - 0.12),
                                 arrowprops=dict(arrowstyle="->", color=GREY, lw=1.2))
                ax_tree.text((x + child_x) / 2, (y_pos + child_y) / 2, answer, ha="center", va="center", fontsize=10,
                             bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor=GREY))

        ax_tree.text(x, y_pos, label, ha="center", va="center", fontsize=11, bbox=box)

    lowest_level = min(y_pos for _, y_pos in positions.values())
    ax_tree.set_xlim(-0.7, len(leaves) - 0.3)
    ax_tree.set_ylim(lowest_level - 0.5, 0.5)
    ax_tree.axis("off")

    # --- правила листьев ---
    ax_rules.axis("off")
    row_height = 1 / len(leaves)

    for row, (node, bounds) in enumerate(leaves):
        row_y = 1 - (row + 0.5) * row_height
        color = CLASS_COLORS[leaf_class[node]]

        badge = plt.Rectangle((0.02, row_y - 0.4 * row_height), 0.12, 0.8 * row_height,
                              transform=ax_rules.transAxes, facecolor=color, alpha=0.3, edgecolor=BLACK)
        ax_rules.add_patch(badge)
        ax_rules.text(0.08, row_y, class_names[leaf_class[node]], transform=ax_rules.transAxes,
                      ha="center", va="center", fontsize=13)

        rule = f"r{row + 1}".translate(SUBSCRIPT) + f"(x) = {rule_text(bounds)}"
        ax_rules.text(0.16, row_y, rule, transform=ax_rules.transAxes, ha="left", va="center", fontsize=14)

    plt.show()


def plot_regions(ax, model, X, y, title, features=FEATURES_2D, resolution=300, legend=False, X_range=None):
    """Области классов модели на плоскости двух признаков и объекты X; пределы осей — по X_range или X."""
    x_name, y_name = features
    X_range = X if X_range is None else X_range

    x_values = np.linspace(X_range[x_name].min() - 0.3, X_range[x_name].max() + 0.3, resolution)
    y_values = np.linspace(X_range[y_name].min() - 0.5, X_range[y_name].max() + 0.5, resolution)
    grid_x, grid_y = np.meshgrid(x_values, y_values)

    grid_points = pd.DataFrame({x_name: grid_x.ravel(), y_name: grid_y.ravel()})
    predicted_class = model.predict(grid_points).reshape(grid_x.shape)

    ax.contourf(grid_x, grid_y, predicted_class, levels=[-0.5, 0.5, 1.5, 2.5], colors=CLASS_COLORS, alpha=0.18)

    for k in range(3):
        is_class = y == k
        ax.scatter(X.loc[is_class, x_name], X.loc[is_class, y_name], s=16, color=CLASS_COLORS[k], label=f"сорт {k}")

    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(title, fontsize=12)
    ax.grid(False)
    if legend:
        ax.legend(loc="upper right", fontsize=10)

## 1. Дерево как набор правил

Данные о винах (`load_wine`): 178 образцов трех сортов, 13 химических
признаков. Для рисунков берут два признака — флавоноиды (`flavanoids`)
и интенсивность цвета (`color_intensity`). Тестовый сет — 30% объектов.

In [ ]:
wine = load_wine(as_frame=True)
X_wine = wine.data
y_wine = wine.target.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=SEED, stratify=y_wine
)

print("обучение:", X_train.shape)
print("тест:", X_test.shape)
print("объектов каждого сорта в обучении:", np.bincount(y_train))

Дерево решений (decision tree) задает вопросы вида «признак ≤ порог?»
и по ответу переходит влево (да) или вправо (нет), пока не дойдет до листа.
Каждому листу соответствует прямоугольная область плоскости признаков,
а предсказание в листе — самый частый сорт среди обучающих объектов,
попавших в эту область. Дерево с четырьмя листьями (`max_leaf_nodes=4`):

In [ ]:
small_tree = DecisionTreeClassifier(max_leaf_nodes=4, random_state=SEED)
small_tree.fit(X_train[FEATURES_2D], y_train)

draw_tree_rules(small_tree, X_train, y_train)

Путь от корня до листа — конъюнкция условий, то есть правило $r_m(x)$, равное 1
для объектов области $R_m$ и 0 для остальных. В каждом листе указано число
обучающих объектов каждого сорта. Два листа, $R_1$ и $R_3$, предсказывают
один сорт: дерево может разбить область одного класса на несколько
прямоугольников.

Предсказание для одного объекта тестового сета — проход по условиям.
Метод `decision_path` возвращает номера узлов на пути объекта от корня
до листа; в последнем узле пути, листе, вопроса нет.

In [ ]:
sample = X_test[FEATURES_2D].iloc[[3]]      # таблица из одной строки
true_class = y_test[3]

structure = small_tree.tree_
path_nodes = small_tree.decision_path(sample).indices
question_nodes = path_nodes[:-1]

for node in question_nodes:
    feature = FEATURES_2D[structure.feature[node]]
    threshold = structure.threshold[node]
    value = sample[feature].iloc[0]

    if value <= threshold:
        answer = "да, влево"
    else:
        answer = "нет, вправо"

    print(f"{feature} = {value:.2f} ≤ {threshold:.2f}?  {answer}")

leaf_shares = small_tree.predict_proba(sample)[0]
predicted_class = small_tree.predict(sample)[0]

print()
print("доли сортов в листе:", leaf_shares.round(2))
print("предсказание: сорт", predicted_class)
print("истинный сорт:", true_class)

## 2. Как дерево выбирает вопрос

Вопрос узла выбирают так, чтобы в каждой из двух получившихся групп почти все
объекты были одного сорта. Неоднородность группы измеряют **индексом Джини**
(Gini impurity) — вероятностью ошибиться, если приписать случайному объекту
группы случайный сорт с вероятностями, равными долям сортов $p_k$. Объект
сорта $k$ получает неверную метку с вероятностью $1 - p_k$, поэтому

$$G = \sum_{k} p_k (1 - p_k) = \sum_k p_k - \sum_k p_k^2 = 1 - \sum_{k} p_k^2.$$

$G = 0$, если все объекты одного сорта, и $G$ наибольший при равных долях;
для трех сортов наибольшее значение $1 - 3 \cdot \frac{1}{9} = \frac{2}{3}$.
Качество вопроса — неоднородность двух получившихся групп, взвешенная
их размерами:

$$G_{\text{после}} = \frac{n_l}{n}\,G_l + \frac{n_r}{n}\,G_r,$$

где $n$ — число объектов узла, $n_l$ и $n_r$ — числа объектов в группах «да» и «нет»,
$G_l$ и $G_r$ — их индексы Джини. Выбирают вопрос с наименьшим $G_{\text{после}}$,
то есть с наибольшим уменьшением неоднородности $G_{\text{до}} - G_{\text{после}}$.

Расчет вручную для корня дерева: 124 обучающих объекта и вопрос
«`color_intensity` ≤ 3.82?».

In [ ]:
def gini(counts):
    shares = np.asarray(counts) / np.sum(counts)
    value = 1 - np.sum(shares ** 2)
    return value


goes_left = (X_train["color_intensity"] <= 3.82).to_numpy()      # True — ответ «да»

root_counts = np.bincount(y_train)
left_counts = np.bincount(y_train[goes_left], minlength=3)
right_counts = np.bincount(y_train[~goes_left], minlength=3)

share_left = goes_left.mean()
share_right = 1 - share_left

gini_root = gini(root_counts)
gini_left = gini(left_counts)
gini_right = gini(right_counts)

gini_after = share_left * gini_left + share_right * gini_right
gini_decrease = gini_root - gini_after

print(f"корень {root_counts.tolist()}: G = {gini_root:.3f}")
print(f"да     {left_counts.tolist()}: G = {gini_left:.3f}, доля объектов {share_left:.3f}")
print(f"нет    {right_counts.tolist()}: G = {gini_right:.3f}, доля объектов {share_right:.3f}")
print()
print(f"после вопроса: {share_left:.3f} · {gini_left:.3f} + {share_right:.3f} · {gini_right:.3f} = {gini_after:.3f}")
print(f"уменьшение неоднородности: {gini_root:.3f} − {gini_after:.3f} = {gini_decrease:.3f}")

Чтобы найти лучший вопрос, перебирают все признаки и все пороги — середины
между соседними значениями признака — и для каждого вычисляют взвешенную
неоднородность. На рисунке она показана для обоих признаков; наименьшее
из двух минимумов — вопрос, который sklearn выбрал для корня.

In [ ]:
def gini_after_question(feature, threshold):
    """Взвешенный индекс Джини двух групп после вопроса «feature ≤ threshold»."""
    goes_left = (X_train[feature] <= threshold).to_numpy()
    share_left = goes_left.mean()

    left_counts = np.bincount(y_train[goes_left], minlength=3)
    right_counts = np.bincount(y_train[~goes_left], minlength=3)

    value = share_left * gini(left_counts) + (1 - share_left) * gini(right_counts)
    return value


fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)

for ax, feature in zip(axes, FEATURES_2D):
    sorted_values = np.sort(X_train[feature].unique())
    thresholds = (sorted_values[1:] + sorted_values[:-1]) / 2         # середины между соседними значениями
    scores = [gini_after_question(feature, threshold) for threshold in thresholds]

    best = int(np.argmin(scores))
    best_threshold = thresholds[best]
    best_score = scores[best]

    ax.plot(thresholds, scores, color=BLUE, lw=2)
    ax.scatter(best_threshold, best_score, s=60, color=OCHRE, zorder=3,
               label=f"минимум: порог {best_threshold:.2f}, G = {best_score:.3f}")
    ax.axhline(gini_root, color=GREY, ls="--", label=f"корень без вопроса: {gini_root:.3f}")
    ax.set_xlabel(f"порог по {feature}")
    ax.legend()

axes[0].set_ylabel("взвешенный индекс Джини")
plt.tight_layout()
plt.show()

Вопрос по `color_intensity` уменьшает неоднородность сильнее, чем лучший
вопрос по `flavanoids`, поэтому он стоит в корне. Затем ту же процедуру
повторяют отдельно в каждой группе. Такой жадный алгоритм называют CART
(classification and regression trees, деревья классификации и регрессии):

1. в узле перебирают все признаки и пороги и выбирают вопрос с наименьшей
   взвешенной неоднородностью;
2. объекты узла делят по ответу;
3. то же повторяют для каждой части, пока не выполнится условие остановки:
   узел чистый, достигнута наибольшая глубина `max_depth`, в узле меньше
   `min_samples_split` объектов или в листе осталось бы меньше
   `min_samples_leaf` объектов.

При параметре `max_leaf_nodes` sklearn строит дерево иначе: каждый раз делит
тот лист, разбиение которого дает наибольшее уменьшение неоднородности,
пока листьев не станет `max_leaf_nodes`; так построено дерево из раздела 1.

Вместо индекса Джини можно использовать энтропию $H = -\sum_k p_k \log_2 p_k$:
`criterion="entropy"`; обычно деревья с этими критериями различаются мало.

## 3. Глубина и переобучение

Без ограничений дерево делит объекты, пока все листья не станут чистыми,
и на обучающей выборке не ошибается. В заголовках — accuracy на обучающей
выборке и на тестовом сете и число листьев.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, depth in zip(axes.ravel(), [1, 2, 4, None]):
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    model.fit(X_train[FEATURES_2D], y_train)

    train_accuracy = model.score(X_train[FEATURES_2D], y_train)
    test_accuracy = model.score(X_test[FEATURES_2D], y_test)
    n_leaves = model.get_n_leaves()

    depth_label = "без ограничений" if depth is None else f"глубина {depth}"
    title = f"{depth_label}: обучение {train_accuracy:.2f}, тест {test_accuracy:.2f}, листьев {n_leaves}"
    plot_regions(ax, model, X_train, y_train, title, legend=depth == 1)

plt.tight_layout()
plt.show()

Глубокое дерево выделяет узкие полосы вокруг отдельных объектов: оно
запоминает обучающую выборку, а на тестовом сете качество не растет.
Кривая accuracy от глубины на всех 13 признаках:

In [ ]:
depth_rows = []

for depth in range(1, 9):
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    cv_accuracy = cross_val_score(model, X_train, y_train, cv=5).mean()

    model.fit(X_train, y_train)
    train_accuracy = model.score(X_train, y_train)
    test_accuracy = model.score(X_test, y_test)

    depth_rows.append({
        "глубина": depth,
        "обучение": train_accuracy,
        "кросс-валидация": cv_accuracy,
        "тест": test_accuracy,
    })

depth_curve = pd.DataFrame(depth_rows).set_index("глубина")

plt.figure(figsize=(8, 4))
for column, color in zip(depth_curve.columns, [GREY, OCHRE, BLUE]):
    plt.plot(depth_curve.index, depth_curve[column], "o-", color=color, label=column)

plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

Accuracy на обучающей выборке растет до 1 к глубине 4, а на кросс-валидации
наибольшая при глубине 3 (0.870) и дальше не растет; на тестовом сете
она не меняется начиная с глубины 3.

Параметры `max_depth`, `min_samples_leaf`, `max_leaf_nodes` ограничивают
рост дерева — это его регуляризация. Подбирают их кросс-валидацией
на обучающей выборке. Сравнение дерева без ограничений, дерева глубины 2
и дерева с параметрами, подобранными по сетке:

In [ ]:
grid = {"max_depth": [1, 2, 3, 4, 5, None], "min_samples_leaf": [1, 2, 5, 10]}

search = GridSearchCV(DecisionTreeClassifier(random_state=SEED), grid, cv=5)
search.fit(X_train, y_train)
best_params = search.best_params_
print("лучшие параметры по кросс-валидации:", best_params)

candidates = {
    "без ограничений": DecisionTreeClassifier(random_state=SEED),
    "max_depth=2": DecisionTreeClassifier(max_depth=2, random_state=SEED),
    "подобранное по сетке": DecisionTreeClassifier(random_state=SEED, **best_params),
}

rows = []
for name, model in candidates.items():
    cv_accuracy = cross_val_score(model, X_train, y_train, cv=5).mean()
    model.fit(X_train, y_train)

    rows.append({
        "дерево": name,
        "листьев": model.get_n_leaves(),
        "обучение": model.score(X_train, y_train),
        "кросс-валидация": cv_accuracy,
        "тест": model.score(X_test, y_test),
    })

comparison_table = pd.DataFrame(rows).set_index("дерево")
show_table(comparison_table)

Дерево глубины 2 недообучено. Подобранное по сетке дерево проще дерева
без ограничений (6 листьев против 8) и немного лучше на кросс-валидации
(0.870 против 0.862), а на тестовом сете они равны (0.963). Различия малы:
одно вино тестового сета — это 0.019 accuracy.

## 4. Важность признаков

Важность признака отвечает на вопрос: насколько модель опирается на этот
признак. У деревьев ее измеряют двумя способами.

**Среднее уменьшение неоднородности** (mean decrease in impurity, MDI).
Каждый вопрос дерева наводит порядок: до вопроса группа смешана сильнее,
чем две группы после него, и индекс Джини уменьшается. Это уменьшение —
польза вопроса. Вопрос в корне касается всех вин, вопрос в глубине дерева —
нескольких, поэтому пользу умножают на долю обучающих объектов, попавших
в узел:

$$\text{польза вопроса} = \frac{\text{объектов в узле}}{\text{объектов в выборке}} \cdot \big(G_{\text{до}} - G_{\text{после}}\big).$$

Важность признака — сумма польз всех вопросов по этому признаку, деленная
на суммарную пользу всех вопросов дерева. Смысл: **какая доля всего порядка,
наведенного деревом, пришлась на этот признак**. Сумма важностей равна 1.

Разбор на дереве с четырьмя листьями из раздела 1: польза каждого из трех
вопросов.

In [ ]:
structure = small_tree.tree_
n_train = structure.n_node_samples[0]
question_rows = []

for node in range(structure.node_count):
    left_child = structure.children_left[node]
    right_child = structure.children_right[node]
    if left_child == -1:
        continue                                    # лист: вопроса нет

    feature = FEATURES_2D[structure.feature[node]]
    threshold = structure.threshold[node]

    n_node = structure.n_node_samples[node]
    n_left = structure.n_node_samples[left_child]
    n_right = structure.n_node_samples[right_child]

    gini_before = structure.impurity[node]
    gini_after = (n_left * structure.impurity[left_child] + n_right * structure.impurity[right_child]) / n_node
    share_in_node = n_node / n_train
    benefit = share_in_node * (gini_before - gini_after)

    question_rows.append({
        "вопрос": f"{feature} ≤ {threshold:.2f}",
        "признак": feature,
        "доля объектов в узле": share_in_node,
        "Джини до": gini_before,
        "Джини после": gini_after,
        "польза": benefit,
    })

questions = pd.DataFrame(question_rows).set_index("вопрос")
questions.round(3)

In [ ]:
benefit_by_feature = questions.groupby("признак")["польза"].sum()
total_benefit = benefit_by_feature.sum()
manual_importance = benefit_by_feature / total_benefit

sklearn_importance = pd.Series(small_tree.feature_importances_, index=FEATURES_2D)

importance_check = pd.DataFrame({
    "сумма польз": benefit_by_feature,
    "важность вручную": manual_importance,
    "feature_importances_": sklearn_importance,
})
importance_check.round(3)

Корневой вопрос по `color_intensity` касается всех вин и дает пользу 0.261.
Признак `flavanoids` стоит в двух вопросах. Первый касается 64% вин
и уменьшает индекс Джини с 0.574 до 0.156 — польза 0.266, даже больше,
чем у корня. Второй уменьшает индекс Джини с 0.231 до 0.161, но касается
только 36% вин, поэтому его польза мала — 0.025.
В сумме на `flavanoids` приходится 0.291 из 0.552 всей пользы дерева,
отсюда важность 0.528, у `color_intensity` — 0.472. Ручной расчет совпадает
с `feature_importances_`.

У MDI два ограничения. Польза считается по обучающей выборке: если дерево
переобучено и делит мелкие узлы по случайным совпадениям, признак получает
пользу, которой на новых данных нет. Кроме того, у признака с большим числом
различных значений больше порогов, и среди них чаще находится порог,
случайно уменьшающий неоднородность на обучении. Пример, где шумовой признак
получает заметную MDI, — в конспекте.

**Перестановочная важность** (permutation importance) проверяет признак
на новых данных. Значения признака в тестовом сете перемешивают между
объектами: сам набор значений остается прежним, но у каждого вина теперь
чужое значение признака, и связь признака с сортом разрушена. Если модель
опирается на признак, она начинает ошибаться; **падение accuracy и есть
важность**. Если модель признак не использует, перемешивание ничего не меняет.

Перемешивание одного признака вручную для дерева, подобранного по сетке:

In [ ]:
best_tree = candidates["подобранное по сетке"]
accuracy_before = best_tree.score(X_test, y_test)

rng_permutation = np.random.default_rng(SEED)
X_test_shuffled = X_test.copy()
shuffled_values = rng_permutation.permutation(X_test["flavanoids"].to_numpy())
X_test_shuffled["flavanoids"] = shuffled_values

accuracy_after = best_tree.score(X_test_shuffled, y_test)
importance_flavanoids = accuracy_before - accuracy_after

print(f"accuracy на тестовом сете:               {accuracy_before:.3f}")
print(f"accuracy после перемешивания flavanoids: {accuracy_after:.3f}")
print(f"перестановочная важность flavanoids:     {importance_flavanoids:.3f}")

Функция `permutation_importance` делает то же для каждого признака
и повторяет перемешивание `n_repeats` раз: результат одного перемешивания
случаен, поэтому берут среднее, а усы на рисунке — стандартное отклонение.
Слева для сравнения MDI того же дерева.

In [ ]:
permutation = permutation_importance(best_tree, X_test, y_test, n_repeats=30, random_state=SEED)

importances = pd.DataFrame({
    "MDI": best_tree.feature_importances_,
    "перестановочная": permutation.importances_mean,
    "std": permutation.importances_std,
}, index=X_wine.columns)
importances = importances.sort_values("MDI")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].barh(importances.index, importances["MDI"], color=BLUE)
axes[0].set_title("MDI: доля порядка, наведенного признаком на обучении")
axes[0].set_xlabel("MDI")

axes[1].barh(importances.index, importances["перестановочная"], xerr=importances["std"], color=OCHRE, capsize=3)
axes[1].set_title("перестановочная: падение accuracy на тестовом сете")
axes[1].set_xlabel("падение accuracy")
axes[1].axvline(0, color=BLACK, lw=0.8)

plt.tight_layout()
plt.show()

Перемешивание `flavanoids` снижает accuracy с 0.963 до 0.593: после него
дерево ошибается еще на 37% тестовых вин; в среднем по 30
перемешиваниям падение 0.315. Дерево использует четыре признака из 13,
у остальных обе важности равны нулю.

Два способа отвечают на разные вопросы, это видно по `proline`. Его MDI
мала — 0.115: вопрос «`proline` ≤ 724.5» стоит в узле, где уже 39 вин сорта 0
и 6 вин сорта 1, и отделяет только эти шесть вин — порядка прибавляется
немного. Но через этот узел проходят почти все вина сорта 0 (39 из 41), и ответ на вопрос
решает, отнести ли вино к сорту 0 или 1. После перемешивания часть вин
сорта 0 получает чужое малое значение `proline`, уходит в лист сорта 1,
и accuracy падает на 0.209. MDI показывает, сколько порядка признак навел
при построении дерева; перестановочная важность — насколько от признака
зависят предсказания на новых данных.

## 5. Пропущенные значения

Если у объекта неизвестен признак, по которому разбит узел, его можно
направить так:

- **спуск в оба поддерева:** объект идет в обоих потомков, и ответ узла —
  среднее ответов поддеревьев с весами, равными долям обучающих объектов
  в потомках: $a(x) = w_l\, a_l(x) + w_r\, a_r(x)$;
- **правило sklearn:** при обучении объекты с пропуском пробуют отправить
  целиком влево и вправо и запоминают лучшее направление; `DecisionTreeClassifier`
  принимает `NaN` без подготовки;
- **заполнение константой вне диапазона**, например $-1$ для положительного
  признака: дерево отделяет такие объекты одним порогом.

Заполнение медианой стирает информацию о том, что значение было пропущено.
Сравнение на винах: в обучающей выборке и на тестовом сете скрывают 30%
значений трех самых важных признаков, 20 случайных масок.

In [ ]:
MISSING_FEATURES = ["flavanoids", "color_intensity", "proline"]
MISSING_SHARE = 0.3


def hide_values(X, rng):
    """Копия таблицы, где значения MISSING_FEATURES заменены на NaN с вероятностью MISSING_SHARE."""
    hidden = X.copy()
    for feature in MISSING_FEATURES:
        is_hidden = rng.random(len(hidden)) < MISSING_SHARE
        hidden.loc[is_hidden, feature] = np.nan
    return hidden


def new_tree():
    model = DecisionTreeClassifier(max_depth=4, random_state=SEED)
    return model


ways = {
    "встроенная обработка sklearn": lambda: new_tree(),
    "заполнение медианой": lambda: make_pipeline(SimpleImputer(strategy="median"), new_tree()),
    "заполнение константой −1": lambda: make_pipeline(SimpleImputer(strategy="constant", fill_value=-1), new_tree()),
}

In [ ]:
missing_scores = {name: [] for name in ways}

for repeat in range(20):
    rng = np.random.default_rng(repeat)
    X_train_missing = hide_values(X_train, rng)
    X_test_missing = hide_values(X_test, rng)

    for name, make_model in ways.items():
        model = make_model()
        model.fit(X_train_missing, y_train)
        missing_scores[name].append(model.score(X_test_missing, y_test))

complete_model = new_tree()
complete_model.fit(X_train, y_train)
print(f"без пропусков: accuracy {complete_model.score(X_test, y_test):.3f}")

mean_scores = {name: np.mean(scores) for name, scores in missing_scores.items()}
missing_table = pd.DataFrame({"accuracy на тестовом сете": mean_scores})
show_table(missing_table)

Встроенная обработка и константа вне диапазона позволяют дереву выучить,
куда направлять объекты с пропуском, и в среднем дают большую accuracy,
чем медиана; различие порядка двух тестовых вин. Сравнение спуска в оба
поддерева с правилом sklearn и суррогатные разбиения — в конспекте, раздел 9.

## 6. Дерево для регрессии

В регрессии лист предсказывает среднее ответов своих объектов, а вопрос
выбирают по уменьшению среднеквадратичной ошибки (mean squared error, MSE).
Предсказание — ступенчатая функция. Данные: $y = \sin x$ плюс нормальный
шум со стандартным отклонением 0.25, 80 точек.

In [ ]:
rng = np.random.default_rng(0)

x_1d = np.sort(rng.uniform(0, 2 * np.pi, 80))[:, None]       # (80, 1): один признак
noise_1d = rng.normal(0, 0.25, len(x_1d))
y_1d = np.sin(x_1d[:, 0]) + noise_1d

x_plot = np.linspace(0, 2 * np.pi, 600)[:, None]

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)

for ax, depth in zip(axes.ravel(), [1, 2, 4, None]):
    model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    model.fit(x_1d, y_1d)

    depth_label = "без ограничений" if depth is None else f"глубина {depth}"
    ax.scatter(x_1d, y_1d, s=14, color=GREY, label="обучающие точки")
    ax.plot(x_plot, np.sin(x_plot), color=OCHRE, lw=1.5, ls="--", label="sin x")
    ax.plot(x_plot, model.predict(x_plot), color=BLUE, lw=2, label="дерево")
    ax.set_title(f"{depth_label}: листьев {model.get_n_leaves()}", fontsize=12)

for ax in axes[1]:
    ax.set_xlabel("x")
for ax in axes[:, 0]:
    ax.set_ylabel("y")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

Дерево без ограничений проходит через каждую обучающую точку и повторяет шум.

**Дерево — линейная модель.** Если области листьев $R_1, \dots, R_M$ уже найдены,
предсказание $a(x) = \sum_m c_m [x \in R_m]$, где $c_m$ — значение листа $m$, линейно
по коэффициентам $c_m$ над новыми признаками — индикаторами листьев $[x \in R_m]$.
Метод наименьших квадратов (МНК) на этих признаках дает в точности средние
ответов листьев (доказательство — в конспекте, раздел 1): нелинейность дерева
целиком в выборе областей.

Проверка на дереве глубины 4 по шагам. Сначала для каждой точки узнают,
в какой лист она попала: метод `apply` возвращает номер листа. sklearn
нумерует все узлы дерева, и внутренние тоже, поэтому номера листьев идут
с пропусками.

In [ ]:
stair_tree = DecisionTreeRegressor(max_depth=4, random_state=SEED)
stair_tree.fit(x_1d, y_1d)

leaf_of_point = stair_tree.apply(x_1d)          # (80,): номер листа для каждой точки
leaf_ids = np.unique(leaf_of_point)             # (M,): номера всех листьев по возрастанию

print("листья первых пяти точек:", leaf_of_point[:5])
print("номера всех листьев:", leaf_ids)
print("листьев:", len(leaf_ids))

Затем строят матрицу индикаторов листьев $\Phi$ размера (число точек,
число листьев): в строке точки $i$ и столбце листа $m$ стоит 1, если точка
попала в лист $m$, и 0 иначе. В каждой строке ровно одна единица: каждая
точка лежит ровно в одном листе.

In [ ]:
n_points = len(leaf_of_point)
n_leaves = len(leaf_ids)
leaf_features = np.zeros((n_points, n_leaves))

for column, leaf_id in enumerate(leaf_ids):
    in_this_leaf = leaf_of_point == leaf_id
    leaf_features[in_this_leaf, column] = 1

print("первые пять строк матрицы индикаторов:")
print(leaf_features[:5].astype(int))
print()
print("единиц в строке:", np.unique(leaf_features.sum(axis=1)))

Наконец, обучают линейную регрессию без свободного члена на столбцах
индикаторов и сравнивают ее коэффициенты со значениями листьев дерева.

In [ ]:
ols = LinearRegression(fit_intercept=False)
ols.fit(leaf_features, y_1d)

leaf_values = stair_tree.tree_.value[leaf_ids, 0, 0]

leaf_comparison = pd.DataFrame({
    "номер листа": leaf_ids,
    "точек в листе": leaf_features.sum(axis=0).astype(int),
    "коэффициент МНК": ols.coef_,
    "значение листа": leaf_values,
})
leaf_comparison.round(3)

Коэффициенты линейной регрессии на индикаторах листьев совпадают со значениями листьев.

## 7. Неустойчивость, смещение и разброс

Небольшое изменение обучающей выборки меняет дерево. Ниже — деревья без
ограничения глубины, обученные на четырех бутстрэп-выборках:
из обучающей выборки случайно с возвращением выбирают столько же объектов.
На каждой панели — объекты той выборки, на которой обучено дерево.

In [ ]:
rng_boot = np.random.default_rng(1)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, number in zip(axes.ravel(), range(1, 5)):
    bootstrap_index = rng_boot.integers(0, len(X_train), len(X_train))     # номера объектов, с повторами
    X_bootstrap = X_train[FEATURES_2D].iloc[bootstrap_index]
    y_bootstrap = y_train[bootstrap_index]

    model = DecisionTreeClassifier(random_state=SEED)
    model.fit(X_bootstrap, y_bootstrap)

    test_accuracy = model.score(X_test[FEATURES_2D], y_test)
    n_unique = len(np.unique(bootstrap_index))
    title = f"выборка {number}: различных объектов {n_unique}, тест {test_accuracy:.2f}, листьев {model.get_n_leaves()}"
    plot_regions(ax, model, X_bootstrap, y_bootstrap, title, legend=number == 1, X_range=X_train)

plt.tight_layout()
plt.show()

Пороги и области заметно меняются от выборки к выборке, хотя все выборки
взяты из одних данных. Это явление описывает **разложение ошибки на смещение
и разброс** (bias-variance decomposition). Пусть $y = f(x) + \varepsilon$, где $f$ — истинная
зависимость, шум имеет нулевое среднее и дисперсию $\sigma_\varepsilon^2$, а $a_D(x)$ — модель,
обученная на случайной выборке $D$. Средняя по выборкам и шуму квадратичная
ошибка в точке $x$:

$$\boxed{\;\mathbb{E}_{D,\varepsilon}\big(y - a_D(x)\big)^2 = \sigma_\varepsilon^2 + \big(f(x) - \mathbb{E}_D\, a_D(x)\big)^2 + \mathbb{E}_D\big(a_D(x) - \mathbb{E}_D\, a_D(x)\big)^2\;}$$

- шум $\sigma_\varepsilon^2$ не зависит от модели;
- смещение (bias) $f(x) - \mathbb{E}_D\, a_D(x)$ — насколько среднее по выборкам предсказание
  отличается от истинной функции; велико у слишком простых моделей;
- разброс (variance) — насколько предсказание меняется от выборки к выборке;
  велик у моделей, которые подстраиваются под выборку.

Вывод формулы — в конспекте, раздел 10. Оценка на синусе: 300 обучающих выборок
по 80 точек, на каждой обучают дерево заданной глубины.

In [ ]:
def simulate_bias_variance(make_model, n_datasets=300, n_train=80, noise=0.25, seed=0):
    """Смещение², разброс и ошибка модели на синусе по многим обучающим выборкам."""
    rng = np.random.default_rng(seed)

    x_eval = np.linspace(0, 2 * np.pi, 200)[:, None]      # (200, 1): точки, где оценивают слагаемые
    f_eval = np.sin(x_eval[:, 0])                         # (200,): истинная функция в этих точках

    predictions = np.empty((n_datasets, len(x_eval)))     # строка b — предсказания модели, обученной на выборке b
    for b in range(n_datasets):
        x_sample = rng.uniform(0, 2 * np.pi, n_train)[:, None]
        noise_sample = rng.normal(0, noise, n_train)
        y_sample = np.sin(x_sample[:, 0]) + noise_sample

        model = make_model()
        model.fit(x_sample, y_sample)
        predictions[b] = model.predict(x_eval)

    mean_prediction = predictions.mean(axis=0)            # (200,): среднее предсказание по выборкам
    bias_squared = np.mean((mean_prediction - f_eval) ** 2)
    variance = np.mean(predictions.var(axis=0))

    y_new = f_eval + rng.normal(0, noise, predictions.shape)
    direct_error = np.mean((y_new - predictions) ** 2)

    estimate = {
        "смещение²": bias_squared,
        "разброс": variance,
        "дисперсия шума": noise ** 2,
        "сумма трех слагаемых": bias_squared + variance + noise ** 2,
        "ошибка напрямую": direct_error,
    }
    return estimate, x_eval, predictions


bv_rows = []
bv_predictions = {}

for depth in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    def make_tree():
        model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
        return model

    estimate, x_eval, predictions = simulate_bias_variance(make_tree)
    bv_rows.append({"глубина": str(depth), **estimate})
    bv_predictions[depth] = predictions

bias_variance = pd.DataFrame(bv_rows).set_index("глубина")
show_table(bias_variance, digits=4, lower_is_better=True)

Сумма трех слагаемых совпадает с ошибкой, посчитанной напрямую. Слева —
деревья глубины 1 на 30 выборках, справа — деревья без ограничения
глубины; черная линия — среднее предсказание по всем 300 выборкам.
Ниже — слагаемые в зависимости от глубины.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, depth in zip(axes, [1, None]):
    predictions = bv_predictions[depth]
    mean_prediction = predictions.mean(axis=0)

    for number, one_dataset_curve in enumerate(predictions[:30]):
        label = "30 деревьев на разных выборках" if number == 0 else None
        ax.plot(x_eval, one_dataset_curve, color=BLUE, lw=0.7, alpha=0.35, label=label)
    ax.plot(x_eval, mean_prediction, color=BLACK, lw=2.2, label="среднее предсказание")
    ax.plot(x_eval, np.sin(x_eval), color=OCHRE, lw=2, ls="--", label="sin x")

    row = bias_variance.loc[str(depth)]
    depth_label = "без ограничений" if depth is None else f"глубина {depth}"
    ax.set_title(f"{depth_label}: смещение² {row['смещение²']:.3f}, разброс {row['разброс']:.3f}", fontsize=12)
    ax.set_xlabel("x")

axes[0].set_ylabel("предсказание дерева")
axes[0].legend()
plt.tight_layout()
plt.show()

depth_ticks = ["∞" if depth == "None" else depth for depth in bias_variance.index]

plt.figure(figsize=(9, 4.2))
for column, color in [("смещение²", BLUE), ("разброс", OCHRE), ("сумма трех слагаемых", BLACK)]:
    plt.plot(depth_ticks, bias_variance[column], "o-", color=color, label=column)
plt.axhline(0.25 ** 2, color=GREY, ls="--", label=r"дисперсия шума $\sigma_\varepsilon^2$")
plt.xlabel("max_depth (∞ — без ограничений)")
plt.ylabel("слагаемые ошибки, среднее по x")
plt.legend()
plt.show()

Неглубокое дерево устойчиво, но не повторяет синус: большое смещение.
Глубокое дерево в среднем повторяет синус, но каждое отдельное дерево
следует за шумом: большой разброс. Лучшая глубина — промежуточная.

## 8. Бутстрэп и бэггинг

Разброс глубоких деревьев уменьшают усреднением деревьев, обученных
на разных выборках; такие выборки получают бутстрэпом.

**Бутстрэп** (bootstrap) — способ получить новую выборку из исходной выборки
размера $n$: $n$ раз выбирают объект случайно и с возвращением. Объект не выбран ни разу с вероятностью
$\big(1 - \frac{1}{n}\big)^n \approx e^{-1} \approx 0.37$, поэтому в бутстрэп-выборку попадает около 63%
различных объектов, а остальные 37% оказываются **вне выборки** (out-of-bag, OOB).

In [ ]:
rng_bootstrap = np.random.default_rng(SEED)
sizes = np.array([2, 3, 5, 10, 20, 50, 100, 200, 500, 1000])

unique_share = []
for n in sizes:
    shares_for_n = []
    for _ in range(300):
        bootstrap_index = rng_bootstrap.integers(0, n, n)
        n_unique = len(np.unique(bootstrap_index))
        shares_for_n.append(n_unique / n)
    unique_share.append(np.mean(shares_for_n))

theory = 1 - (1 - 1 / sizes) ** sizes
limit = 1 - np.exp(-1)

plt.figure(figsize=(8, 3.8))
plt.semilogx(sizes, unique_share, "o", color=BLUE, label="доля различных объектов, 300 выборок")
plt.semilogx(sizes, theory, color=BLACK, lw=1.5, label="1 − (1 − 1/n)ⁿ")
plt.axhline(limit, color=OCHRE, ls="--", label="1 − 1/e ≈ 0.632")
plt.xlabel("размер выборки n")
plt.ylabel("доля")
plt.legend()
plt.show()

**Бэггинг** (bagging, bootstrap aggregating) обучает $B$ моделей на $B$ бутстрэп-выборках
и усредняет их предсказания: $a(x) = \frac{1}{B}\sum_{k=1}^{B} b_k(x)$.

Насколько уменьшается разброс, зависит от того, насколько похожи модели
ансамбля. В фиксированной точке $x$ вводят две величины:

- $\sigma^2$ — разброс одной модели: дисперсия $\mathbb{D}$ ее предсказания $b_k(x)$ при смене
  обучающей выборки и бутстрэп-выборки, как в разделе 7;
- $\rho$ — коэффициент корреляции предсказаний двух разных моделей ансамбля
  в этой точке: $\rho = \operatorname{cov}\big(b_k(x), b_l(x)\big) / \sigma^2$ при $k \ne l$.

Модели обучены на бутстрэп-выборках из одних и тех же данных и видят во многом
одни и те же объекты, поэтому их предсказания похожи и $\rho > 0$. Величину $\rho$
не задают заранее: это свойство базового алгоритма и данных. Ее оценивают:
на многих обучающих выборках обучают по две модели на разных бутстрэп-выборках
и считают выборочный коэффициент корреляции их предсказаний. $\rho$ уменьшают,
добавляя случайность в построение моделей, — так устроен случайный лес
из раздела 9.

**Смещение ансамбля.** Все модели обучены одним алгоритмом на одинаково
устроенных случайных выборках, поэтому у них одно и то же среднее предсказание
$\mathbb{E}\, b_k(x) = \mathbb{E}\, b_1(x)$. По линейности математического ожидания

$$\mathbb{E}\, a(x) = \mathbb{E}\Big(\frac{1}{B}\sum_{k=1}^{B} b_k(x)\Big) = \frac{1}{B}\sum_{k=1}^{B} \mathbb{E}\, b_k(x) = \mathbb{E}\, b_1(x),$$

то есть среднее предсказание ансамбля совпадает со средним одной модели,
обученной на бутстрэп-выборке, и смещение $f(x) - \mathbb{E}\, a(x)$ у них одинаковое.

**Разброс ансамбля.** Константа $\frac{1}{B}$ выносится из дисперсии в квадрате, а дисперсия
суммы равна сумме дисперсий слагаемых плюс сумма ковариаций всех пар:

$$\mathbb{D}\, a(x) = \frac{1}{B^2}\,\mathbb{D}\Big(\sum_{k=1}^{B} b_k(x)\Big) = \frac{1}{B^2}\Big(\sum_{k=1}^{B} \mathbb{D}\, b_k(x) + \sum_{k \ne l} \operatorname{cov}\big(b_k(x), b_l(x)\big)\Big).$$

В первой сумме $B$ слагаемых, каждое равно $\sigma^2$; во второй — $B(B - 1)$ упорядоченных
пар, каждое слагаемое равно $\rho\sigma^2$:

$$\mathbb{D}\, a(x) = \frac{B\sigma^2 + B(B - 1)\,\rho\sigma^2}{B^2} = \frac{\sigma^2}{B} + \frac{B - 1}{B}\,\rho\sigma^2.$$

Слагаемое $\frac{B - 1}{B}\,\rho\sigma^2 = \rho\sigma^2 - \frac{\rho\sigma^2}{B}$; объединение двух слагаемых с множителем $\frac{1}{B}$ дает

$$\boxed{\;\mathbb{D}\, a(x) = \rho\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2\;}$$

При $\rho = 0$ разброс уменьшается в $B$ раз, при $\rho = 1$ не уменьшается, а при росте $B$
приближается к пределу $\rho\sigma^2$, ниже которого не опускается. Бэггинг выгоден
для моделей с малым смещением и большим разбросом — глубоких деревьев.

Оценка $\sigma^2$ и $\rho$ для деревьев без ограничения глубины на синусе: 300 обучающих
выборок по 80 точек, на каждой — два дерева на двух бутстрэп-выборках.

In [ ]:
rng_pairs = np.random.default_rng(7)
n_datasets = 300
x_eval_pairs = np.linspace(0, 2 * np.pi, 200)[:, None]
pair_predictions = np.empty((n_datasets, 2, len(x_eval_pairs)))     # [выборка, дерево, точка]

for d in range(n_datasets):
    x_sample = rng_pairs.uniform(0, 2 * np.pi, 80)[:, None]
    noise_sample = rng_pairs.normal(0, 0.25, 80)
    y_sample = np.sin(x_sample[:, 0]) + noise_sample

    for k in range(2):
        bootstrap_index = rng_pairs.integers(0, 80, 80)
        tree = DecisionTreeRegressor(random_state=SEED)
        tree.fit(x_sample[bootstrap_index], y_sample[bootstrap_index])
        pair_predictions[d, k] = tree.predict(x_eval_pairs)

first_tree = pair_predictions[:, 0]             # (300, 200): первое дерево на каждой выборке
second_tree = pair_predictions[:, 1]            # (300, 200): второе дерево на тех же выборках

sigma2 = (first_tree.var(axis=0) + second_tree.var(axis=0)) / 2      # разброс одного дерева в каждой точке

first_centered = first_tree - first_tree.mean(axis=0)
second_centered = second_tree - second_tree.mean(axis=0)
covariance = (first_centered * second_centered).mean(axis=0)
rho = covariance / sigma2                       # коэффициент корреляции в каждой точке

variance_floor = np.mean(rho * sigma2)
print(f"разброс одного дерева σ² (среднее по точкам): {sigma2.mean():.4f}")
print(f"корреляция двух деревьев ρ (среднее по точкам): {rho.mean():.3f}")
print(f"предел разброса бэггинга ρσ²: {variance_floor:.4f}")

Предсказания двух деревьев на бутстрэп-выборках из одних данных заметно
похожи: $\rho \approx 0.45$. Предел разброса $\rho\sigma^2 \approx 0.030$ — меньше половины разброса
одного дерева $\sigma^2 \approx 0.066$: бэггинг из любого числа таких деревьев уменьшает
разброс не больше чем в 2.2 раза.

Проверка формулы: бэггинг из $B$ деревьев, разброс ансамбля по 300 выборкам
и кривая $\rho\sigma^2 + (1 - \rho)\sigma^2 / B$ с оценками $\sigma^2$ и $\rho$ из предыдущей ячейки.

In [ ]:
bagging_rows = []

for B in [1, 2, 5, 10, 25, 50]:
    def make_bagging():
        model = BaggingRegressor(DecisionTreeRegressor(), n_estimators=B, random_state=SEED)
        return model

    estimate, _, _ = simulate_bias_variance(make_bagging)
    bagging_rows.append({"B": B, **estimate})

bagging_table = pd.DataFrame(bagging_rows).set_index("B")
best_single_tree_error = bias_variance.loc["4", "сумма трех слагаемых"]

B_grid = np.arange(1, 51)
formula_variance = [np.mean(rho * sigma2 + (1 - rho) * sigma2 / B) for B in B_grid]

plt.figure(figsize=(9, 4.5))
for column, color in [("смещение²", BLUE), ("разброс", OCHRE), ("сумма трех слагаемых", BLACK)]:
    plt.semilogx(bagging_table.index, bagging_table[column], "o-", color=color, label=column)
plt.semilogx(B_grid, formula_variance, color=OCHRE, ls="--", lw=1.5, label="разброс по формуле ρσ² + (1 − ρ)σ²/B")
plt.axhline(best_single_tree_error, color=GREY, ls=":", label="лучшее одно дерево, глубина 4")
plt.xticks(bagging_table.index, bagging_table.index)
plt.minorticks_off()
plt.xlabel("число деревьев B")
plt.ylabel("слагаемые ошибки, среднее по x")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

show_table(bagging_table[["смещение²", "разброс", "сумма трех слагаемых"]], digits=4, lower_is_better=True)

Смещение² от числа деревьев не зависит, а разброс уменьшается с 0.067
при $B = 1$ до 0.032 при $B = 50$, идет вдоль кривой формулы и приближается
к пределу $\rho\sigma^2 \approx 0.030$. При $B \ge 25$ ансамбль
деревьев без ограничения глубины ошибается так же, как лучшее одно дерево
глубины 4 (0.095 против 0.096), хотя глубину для ансамбля не подбирали.

Бэггинг на винах: области классов одного дерева без ограничений и бэггинга
из 100 таких деревьев на двух признаках.

In [ ]:
single_tree = DecisionTreeClassifier(random_state=SEED)
single_tree.fit(X_train[FEATURES_2D], y_train)

bagging_2d = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=SEED)
bagging_2d.fit(X_train[FEATURES_2D], y_train)

single_accuracy = single_tree.score(X_test[FEATURES_2D], y_test)
bagging_accuracy = bagging_2d.score(X_test[FEATURES_2D], y_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_regions(axes[0], single_tree, X_train, y_train, f"одно дерево: тест {single_accuracy:.3f}", legend=True)
plot_regions(axes[1], bagging_2d, X_train, y_train, f"бэггинг 100 деревьев: тест {bagging_accuracy:.3f}")
plt.tight_layout()
plt.show()

На двух признаках бэггинг повышает accuracy на тестовом сете с 0.870 до 0.907.
Границы областей по-прежнему состоят из отрезков, параллельных осям, но ступени
мельче: ансамбль усредняет много кусочно-постоянных функций.

### Оценка вне выборки

Чтобы оценить качество модели честно, ее проверяют на объектах, которых она
не видела при обучении: на тестовом сете или кросс-валидацией. У бэггинга
такие объекты есть без дополнительных затрат. Каждое дерево $b_k$ обучено на своей бутстрэп-выборке
$X_k$, и около 37% обучающих объектов в нее не попали, как показано выше: для этого
дерева они как новые данные.

Для объекта $x_i$ обучающей выборки берут только деревья, которые его не видели,

$$K_i = \{k : x_i \notin X_k\}, \qquad |K_i| \approx 0.37\,B,$$

и усредняют их предсказания — для классификации векторы вероятностей:

$$a_{\text{OOB}}(x_i) = \frac{1}{|K_i|}\sum_{k \in K_i} b_k(x_i).$$

**Оценка вне выборки** — качество этих предсказаний на всей обучающей выборке,
для классификации — accuracy:

$$\text{OOB} = \frac{1}{n}\sum_{i=1}^{n}\big[\arg\max_j\, a_{\text{OOB}}(x_i)_j = y_i\big],$$

где $j$ пробегает классы. В отличие от кросс-валидации, ничего не нужно обучать
заново: нужные деревья уже обучены, и оценку вычисляют при обучении ансамбля
без дополнительных моделей (`oob_score=True`).

Ограничения следуют из формулы. Предсказание $a_{\text{OOB}}(x_i)$ — среднее только
$|K_i| \approx 0.37B$ деревьев, а не всех $B$, поэтому оценка немного пессимистична,
и при малом $B$ сильно: у части объектов в $K_i$ одно-два дерева. Если объект
попал во все бутстрэп-выборки, $K_i$ пусто, и объект оценить нельзя.

Расчет вручную для бэггинга из 50 деревьев. Атрибут `estimators_samples_`
хранит номера объектов бутстрэп-выборки каждого дерева.

In [ ]:
oob_model = BaggingClassifier(DecisionTreeClassifier(), n_estimators=50, oob_score=True, random_state=SEED)
oob_model.fit(X_train, y_train)

X_train_values = X_train.to_numpy()
n_objects = len(X_train_values)
probability_sum = np.zeros((n_objects, 3))           # сумма вероятностей от деревьев, не видевших объект
trees_not_seen = np.zeros(n_objects, dtype=int)      # |K_i| — сколько деревьев не видели объект i

for tree, sample_index in zip(oob_model.estimators_, oob_model.estimators_samples_):
    in_bootstrap = np.zeros(n_objects, dtype=bool)
    in_bootstrap[sample_index] = True                # объекты бутстрэп-выборки этого дерева
    out_of_bag = ~in_bootstrap                       # объекты, которых дерево не видело

    tree_probabilities = tree.predict_proba(X_train_values[out_of_bag])
    probability_sum[out_of_bag] += tree_probabilities
    trees_not_seen[out_of_bag] += 1

oob_probabilities = probability_sum / trees_not_seen[:, None]
oob_prediction = oob_probabilities.argmax(axis=1)
manual_oob_accuracy = np.mean(oob_prediction == y_train)

share_not_in_bootstrap = (1 - 1 / n_objects) ** n_objects       # вероятность не попасть в бутстрэп-выборку
expected_not_seen = 50 * share_not_in_bootstrap

print(f"деревьев, не видевших объект: в среднем {trees_not_seen.mean():.1f} из 50, ожидание {expected_not_seen:.1f}")
print(f"меньше всего: {trees_not_seen.min()}, больше всего: {trees_not_seen.max()}")
print(f"оценка вне выборки вручную: {manual_oob_accuracy:.3f}")
print(f"oob_score_ из sklearn:      {oob_model.oob_score_:.3f}")

Ручной расчет совпадает с `oob_score_`. Каждый объект оценен в среднем
18 деревьями из 50, как и предсказывает вероятность $(1 - 1/124)^{124} \approx 0.366$.

Оценка вне выборки при разном числе деревьев и для сравнения accuracy
кросс-валидации и на тестовом сете:

In [ ]:
curve_rows = []

for B in [1, 5, 10, 20, 50, 100, 200]:
    has_oob = B >= 10                     # при малом B у части объектов нет деревьев, где они вне выборки
    model = BaggingClassifier(DecisionTreeClassifier(), n_estimators=B, oob_score=has_oob, random_state=SEED)
    cv_accuracy = cross_val_score(model, X_train, y_train, cv=5).mean()

    model.fit(X_train, y_train)
    test_accuracy = model.score(X_test, y_test)
    oob_accuracy = model.oob_score_ if has_oob else np.nan

    curve_rows.append({
        "B": B,
        "тест": test_accuracy,
        "вне выборки (OOB)": oob_accuracy,
        "кросс-валидация": cv_accuracy,
    })

bagging_curve = pd.DataFrame(curve_rows).set_index("B")
tree_cv = cross_val_score(DecisionTreeClassifier(random_state=SEED), X_train, y_train, cv=5).mean()

plt.figure(figsize=(9, 4.5))
for column, color in zip(bagging_curve.columns, [BLUE, BLACK, OCHRE]):
    plt.semilogx(bagging_curve.index, bagging_curve[column], "o-", color=color, label=column)
plt.axhline(tree_cv, color=GREY, ls="--", label=f"одно дерево, кросс-валидация {tree_cv:.3f}")
plt.xticks(bagging_curve.index, bagging_curve.index)
plt.minorticks_off()
plt.xlabel("число деревьев B")
plt.ylabel("accuracy")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

На всех признаках accuracy кросс-валидации
у одного дерева 0.862, а у бэггинга при $B \ge 5$ колеблется от 0.92 до 0.95
и при 100–200 деревьях равна 0.935; оценка вне выборки при $B \ge 50$
совпадает с кросс-валидацией в пределах 0.01, а при малом $B$ занижена:
у части объектов мало деревьев, в обучение которых они не попали.
На тестовом сете из 54 объектов accuracy равна 1.000 уже при $B = 20$,
но одно вино — это 0.019, и такие различия в пределах шума.

## 9. Случайный лес

У бэггинга деревьев остается предел разброса $\rho\sigma^2$: деревья обучены на похожих
выборках, задают похожие вопросы и ошибаются одинаково. Чтобы уменьшить $\rho$,
в обучение добавляют вторую случайность — в выборе признаков.

**Метод случайных подпространств** (random subspace method, RSM; Ho, 1998).
Каждую модель обучают не на всех $p$ признаках, а на случайном подмножестве
из $m < p$ признаков. Это тот же прием, что в бэггинге, только случайно выбирают
столбцы таблицы, а не строки: модели, которые видят разные признаки, ошибаются
по-разному.

**Случайный лес** (random forest; Breiman, 2001) объединяет оба приема,
бэггинг по объектам и метод случайных подпространств по признакам:

1. для $k = 1, \dots, B$ строят бутстрэп-выборку $X_k$ — это бэггинг;
2. на $X_k$ растят дерево без ограничения глубины; в каждом узле заново
   случайно выбирают $m$ признаков из $p$ и ищут лучшее разбиение только по ним — это RSM;
3. ответ леса — среднее предсказаний деревьев, для классификации — среднее
   векторов вероятностей.

Отличие от исходного RSM — выбор признаков в каждом узле, а не один раз
на дерево: сильный признак недоступен в одних узлах, и там дерево задает
другой вопрос, но в других узлах может его использовать.

Параметр $m$ — `max_features`. При $m = p$ случайности в признаках нет, и лес
совпадает с бэггингом деревьев. По умолчанию в sklearn $m = \lfloor\sqrt{p}\rfloor$ для классификации
и $m = p$ для регрессии; Брейман для регрессии рекомендовал $m = p/3$.

Четыре способа на винах в одинаковых условиях, все из 300 деревьев без ограничения
глубины: бэггинг (случайные объекты), RSM (случайные 3 признака из 13 на дерево,
все объекты), бэггинг вместе с RSM на дерево и случайный лес (бэггинг и случайные
3 признака из 13 в каждом узле). Метрика — accuracy кросс-валидации на обучающей
выборке.

In [ ]:
ensembles = {
    "одно дерево": DecisionTreeClassifier(random_state=SEED),
    "бэггинг: случайные объекты": BaggingClassifier(
        DecisionTreeClassifier(), n_estimators=300, random_state=SEED),
    "RSM: случайные признаки на дерево": BaggingClassifier(
        DecisionTreeClassifier(), n_estimators=300, bootstrap=False, max_features=3, random_state=SEED),
    "бэггинг + RSM на дерево": BaggingClassifier(
        DecisionTreeClassifier(), n_estimators=300, max_features=3, random_state=SEED),
    "случайный лес: бэггинг + RSM в каждом узле": RandomForestClassifier(
        n_estimators=300, random_state=SEED),
}

variant_rows = []
for name, model in ensembles.items():
    cv_accuracy = cross_val_score(model, X_train, y_train, cv=5).mean()
    variant_rows.append({"ансамбль": name, "accuracy кросс-валидации": cv_accuracy})

variant_table = pd.DataFrame(variant_rows).set_index("ансамбль")
show_table(variant_table)

Каждая из двух случайностей помогает. Бэггинг поднимает accuracy с 0.862
у одного дерева до 0.935, случайные признаки — до 0.975–0.984. При одинаковом
числе признаков $m = 3$ выбор на дерево (0.984) и в каждом узле (0.975) на винах
дают одинаковое качество в пределах шума: разница — один объект из 124.

Причину выигрыша показывает устройство самих деревьев. Для каждого ансамбля
из 200 деревьев: какой признак стоит в корне; accuracy одного дерева на объектах
вне его бутстрэп-выборки; accuracy всего ансамбля вне выборки; как часто два
дерева дают один ответ на тестовом сете (первые 50 деревьев, все пары).

In [ ]:
bagging_trees = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, oob_score=True, random_state=SEED)
bagging_trees.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=SEED)
forest.fit(X_train, y_train)

X_train_values = X_train.to_numpy()
X_test_values = X_test.to_numpy()
diversity_rows = []
root_shares = {}

for name, ensemble in [("бэггинг", bagging_trees), ("случайный лес", forest)]:
    trees = ensemble.estimators_
    root_features = [X_wine.columns[tree.tree_.feature[0]] for tree in trees]
    root_shares[name] = pd.Series(root_features).value_counts(normalize=True)

    tree_oob_accuracy = []                                    # каждое дерево — на объектах вне своей выборки
    for tree, sample_index in zip(trees, ensemble.estimators_samples_):
        out_of_bag = np.ones(len(X_train_values), dtype=bool)
        out_of_bag[sample_index] = False
        tree_predictions = tree.predict(X_train_values[out_of_bag])
        tree_oob_accuracy.append(np.mean(tree_predictions == y_train[out_of_bag]))

    test_predictions = np.array([tree.predict(X_test_values) for tree in trees])     # (деревья, объекты)
    agreement = []
    for i in range(50):
        for j in range(i + 1, 50):
            same_answer = test_predictions[i] == test_predictions[j]
            agreement.append(same_answer.mean())

    diversity_rows.append({
        "ансамбль": name,
        "accuracy одного дерева вне выборки": np.mean(tree_oob_accuracy),
        "accuracy ансамбля вне выборки": ensemble.oob_score_,
        "совпадение ответов двух деревьев на тестовом сете": np.mean(agreement),
    })

diversity_table = pd.DataFrame(diversity_rows).set_index("ансамбль")
root_table = pd.DataFrame(root_shares).fillna(0).sort_values("бэггинг", ascending=False)

positions = np.arange(len(root_table))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(positions - 0.2, root_table["бэггинг"], width=0.4, color=GREY, label="бэггинг")
ax.bar(positions + 0.2, root_table["случайный лес"], width=0.4, color=BLUE, label="случайный лес")
ax.set_xticks(positions, root_table.index, rotation=45, ha="right")
ax.set_ylabel("доля деревьев")
ax.grid(axis="x", visible=False)
ax.set_title("признак, по которому задан вопрос в корне дерева")
ax.legend()
plt.show()

diversity_table.round(3)

В бэггинге 52% деревьев начинают с вопроса по `color_intensity` — самому
сильному признаку, и деревья похожи друг на друга. В лесу `color_intensity`
попадает в число трех доступных в корне признаков с вероятностью 3/13 ≈ 0.23
и стоит в корне у 15% деревьев; корни распределены почти поровну между четырьмя
признаками (`flavanoids`, `proline`, `alcohol`, `color_intensity`). Каждое дерево
леса по отдельности хуже дерева бэггинга: accuracy вне выборки 0.866 против 0.890,
ему чаще недоступен лучший признак. Но деревья леса реже дают один и тот же ответ
(0.820 против 0.866) и ошибаются по-разному, поэтому их среднее точнее:
0.976 против 0.944 вне выборки. Доля совпадающих ответов косвенно показывает
уменьшение $\rho$: лес жертвует качеством отдельного дерева ради непохожести деревьев.

**Как `max_features` меняет $\rho$.** На винах нельзя получить много независимых
обучающих выборок, поэтому $\rho$ оценивают на синтетических данных Фридмана
(Friedman, 1991): 10 признаков, равномерных на $[0, 1]$, ответ

$$y = 10\sin(\pi x_1 x_2) + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 1),$$

от признаков $x_6, \dots, x_{10}$ ответ не зависит. Как в разделе 8: 200 обучающих
выборок по 500 объектов, на каждой — два дерева леса с заданным $m$. По ним
оценивают разброс одного дерева $\sigma^2$, корреляцию $\rho$, предел разброса $\rho\sigma^2$
и смещение²; ошибка бесконечного леса — шум плюс смещение² плюс $\rho\sigma^2$.

In [ ]:
def friedman_truth(X):
    """Истинная зависимость Фридмана: от признаков с шестого по десятый ответ не зависит."""
    wave = 10 * np.sin(np.pi * X[:, 0] * X[:, 1])
    parabola = 20 * (X[:, 2] - 0.5) ** 2
    linear = 10 * X[:, 3] + 5 * X[:, 4]
    value = wave + parabola + linear
    return value


rng_forest = np.random.default_rng(3)
X_eval_friedman = rng_forest.uniform(0, 1, (300, 10))        # точки, где оценивают слагаемые
f_eval_friedman = friedman_truth(X_eval_friedman)

n_friedman_datasets = 200
n_friedman_objects = 500
friedman_noise = 1.0
friedman_datasets = []
for _ in range(n_friedman_datasets):
    X_sample = rng_forest.uniform(0, 1, (n_friedman_objects, 10))
    noise_sample = rng_forest.normal(0, friedman_noise, n_friedman_objects)
    y_sample = friedman_truth(X_sample) + noise_sample
    friedman_datasets.append((X_sample, y_sample))

forest_rows = []
for m in [1, 2, 3, 5, 7, 10]:                                 # m = 10 — все признаки, то есть бэггинг
    pair_predictions = np.empty((n_friedman_datasets, 2, len(X_eval_friedman)))

    for d, (X_sample, y_sample) in enumerate(friedman_datasets):
        for k in range(2):
            bootstrap_index = rng_forest.integers(0, n_friedman_objects, n_friedman_objects)
            tree_seed = int(rng_forest.integers(1_000_000))      # своя случайность выбора признаков в узлах
            tree = DecisionTreeRegressor(max_features=m, random_state=tree_seed)
            tree.fit(X_sample[bootstrap_index], y_sample[bootstrap_index])
            pair_predictions[d, k] = tree.predict(X_eval_friedman)

    mean_prediction = pair_predictions.mean(axis=(0, 1))
    sigma2_m = pair_predictions.var(axis=(0, 1))
    centered = pair_predictions - mean_prediction
    covariance_m = (centered[:, 0] * centered[:, 1]).mean(axis=0)
    rho_m = covariance_m / sigma2_m

    bias_squared = np.mean((mean_prediction - f_eval_friedman) ** 2)
    floor = np.mean(rho_m * sigma2_m)

    forest_rows.append({
        "max_features": m,
        "разброс дерева σ²": sigma2_m.mean(),
        "корреляция ρ": rho_m.mean(),
        "предел разброса ρσ²": floor,
        "смещение²": bias_squared,
        "ошибка леса при B → ∞": friedman_noise ** 2 + bias_squared + floor,
    })

forest_table = pd.DataFrame(forest_rows).set_index("max_features")
show_table(forest_table, digits=3, lower_is_better=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(forest_table.index, forest_table["корреляция ρ"], "o-", color=BLUE)
axes[0].set_xlabel("max_features m")
axes[0].set_ylabel("ρ")
axes[0].set_title("корреляция предсказаний двух деревьев")

for column, color in [("предел разброса ρσ²", OCHRE), ("смещение²", BLUE), ("ошибка леса при B → ∞", BLACK)]:
    axes[1].plot(forest_table.index, forest_table[column], "o-", color=color, label=column)
axes[1].set_xlabel("max_features m")
axes[1].set_ylabel("средний квадрат ошибки")
axes[1].set_title("слагаемые ошибки бесконечного леса")
axes[1].legend()

for ax in axes:
    ax.set_xticks(forest_table.index)

plt.tight_layout()
plt.show()

С уменьшением $m$ деревья становятся менее похожими: $\rho \approx 0.10$ у бэггинга
($m = 10$) и 0.015 при $m = 1$, предел разброса $\rho\sigma^2$ падает с 0.84 до 0.24–0.29.
Цена — качество отдельного дерева: при малом $m$ оно чаще задает вопросы
по бесполезным признакам $x_6, \dots, x_{10}$, разброс одного дерева $\sigma^2$ растет
с 8.3 до 19.2, а смещение² — с 3.2 до 8.4. Ошибка бесконечного леса наименьшая
при $m = 7$: 4.83 против 5.08 у бэггинга, а при $m = 1$ лес хуже бэггинга почти
вдвое (9.73). На этих данных основная часть ошибки — смещение, и выигрыш
от уменьшения $\rho$ небольшой; `max_features` поэтому подбирают, например,
по оценке вне выборки.

На винах: accuracy вне выборки от числа деревьев для леса и бэггинга и от `max_features`.

In [ ]:
curve_rows = []
for B in [10, 20, 50, 100, 200, 500]:
    forest_b = RandomForestClassifier(n_estimators=B, oob_score=True, random_state=SEED)
    forest_b.fit(X_train, y_train)

    bagging_b = BaggingClassifier(DecisionTreeClassifier(), n_estimators=B, oob_score=True, random_state=SEED)
    bagging_b.fit(X_train, y_train)

    curve_rows.append({"B": B, "случайный лес": forest_b.oob_score_, "бэггинг": bagging_b.oob_score_})

forest_curve = pd.DataFrame(curve_rows).set_index("B")

m_rows = []
for m in [1, 2, 3, 6, 13]:
    forest_m = RandomForestClassifier(n_estimators=300, max_features=m, oob_score=True, random_state=SEED)
    forest_m.fit(X_train, y_train)
    m_rows.append({"max_features": m, "accuracy вне выборки": forest_m.oob_score_})

m_table = pd.DataFrame(m_rows).set_index("max_features")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for column, color in [("случайный лес", BLUE), ("бэггинг", GREY)]:
    axes[0].semilogx(forest_curve.index, forest_curve[column], "o-", color=color, label=column)
axes[0].set_xticks(forest_curve.index, forest_curve.index)
axes[0].minorticks_off()
axes[0].set_xlabel("число деревьев B")
axes[0].set_ylabel("accuracy вне выборки")
axes[0].legend()

axes[1].plot(m_table.index, m_table["accuracy вне выборки"], "o-", color=BLUE)
axes[1].set_xticks(m_table.index)
axes[1].axvline(3, color=GREY, ls="--", label="по умолчанию: ⌊√13⌋ = 3")
axes[1].set_xlabel("max_features m (m = 13 — бэггинг)")
axes[1].set_ylabel("accuracy вне выборки")
axes[1].legend()

plt.tight_layout()
plt.show()

print("accuracy вне выборки от числа деревьев:")
print(forest_curve.T.round(3))
print()
print("accuracy вне выборки от max_features, 300 деревьев:")
print(m_table.T.round(3))

Лес лучше бэггинга при любом числе деревьев и выходит на 0.976 к 100
деревьям; дальнейший рост числа деревьев качество не меняет. По `max_features`
лучшие $m = 1$ и $m = 2$ (0.984), значение по умолчанию $m = 3$ дает 0.976, а $m = 13$,
то есть бэггинг, — 0.935. Значения $m$ от 1 до 6 различаются на один объект
из 124, а переход от бэггинга к лесу дает 5 объектов; в отличие от данных
Фридмана, на винах малое $m$ не ухудшает качество.

**Важность признаков в лесу** — MDI, усредненная по деревьям. Лес распределяет
важность между похожими признаками: если в узле недоступен один признак,
дерево задает вопрос по другому, который несет ту же информацию. Похожесть
признаков видна по матрице корреляций; после нее — MDI бэггинга и леса
из 200 деревьев.

In [ ]:
correlation = X_train.corr()
n_features = len(correlation)

fig, ax = plt.subplots(figsize=(9, 7.5))
image = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)

for i in range(n_features):
    for j in range(n_features):
        value = correlation.iloc[i, j]
        text_color = "white" if abs(value) > 0.6 else BLACK
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8, color=text_color)

ax.set_xticks(range(n_features), correlation.columns, rotation=90)
ax.set_yticks(range(n_features), correlation.columns)
ax.grid(False)
ax.set_title("корреляция признаков вин на обучающей выборке")
fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()

In [ ]:
bagging_importance = np.mean([tree.feature_importances_ for tree in bagging_trees.estimators_], axis=0)

forest_importance = pd.DataFrame({
    "бэггинг": bagging_importance,
    "случайный лес": forest.feature_importances_,
}, index=X_wine.columns)
forest_importance = forest_importance.sort_values("случайный лес")

positions = np.arange(len(forest_importance))
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(positions + 0.2, forest_importance["бэггинг"], height=0.4, color=GREY, label="бэггинг, 200 деревьев")
ax.barh(positions - 0.2, forest_importance["случайный лес"], height=0.4, color=BLUE, label="случайный лес, 200 деревьев")
ax.set_yticks(positions, forest_importance.index)
ax.set_xlabel("MDI")
ax.grid(axis="y", visible=False)
ax.legend()
plt.show()

В бэггинге из тех же 200 деревьев почти вся MDI приходится на `color_intensity`
(0.300), `flavanoids` (0.290) и `proline` (0.168). В лесу важность распределена
шире: `color_intensity` 0.165, `flavanoids` 0.155, `alcohol` 0.146, `proline` 0.143,
затем `od280/od315_of_diluted_wines` 0.095, `hue` 0.094 и `total_phenols` 0.052 против 0.002
в бэггинге. Признак `flavanoids` сильно коррелирован с `total_phenols` (0.86)
и `od280/od315_of_diluted_wines` (0.77): в узлах, где `flavanoids` недоступен, деревья
спрашивают о них, и важность делится между признаками группы. Поэтому важность
в лесу показывает все полезные признаки, но каждый признак из группы
коррелированных получает только часть важности группы.

Еще сильнее деревья разнообразят **крайне случайные деревья** (extremely randomized
trees; Geurts и соавторы, 2006; `ExtraTreesClassifier`): в каждом узле для каждого
из $m$ признаков берут один случайный порог вместо лучшего. Деревья становятся
еще менее похожими и быстрее строятся.

### Границы kNN и деревьев

Метод k ближайших соседей (k nearest neighbors, kNN) с первого занятия, одно
дерево, бэггинг и случайный лес на трех наборах точек: две луны, два кольца
и классы с линейной границей, размытой шумом. Цвет фона — вероятность класса 1,
которую предсказывает модель; точки с белой обводкой — обучающая выборка,
с черной — тестовый сет; число в углу — accuracy на тестовом сете.

In [ ]:
X_linear, y_linear = make_classification(n_samples=300, n_features=2, n_redundant=0, n_informative=2,
                                         n_clusters_per_class=1, random_state=1)
X_linear = X_linear + 2 * np.random.default_rng(2).uniform(size=X_linear.shape)     # размывают границу
toy_datasets = {
    "луны": make_moons(n_samples=300, noise=0.3, random_state=0),
    "кольца": make_circles(n_samples=300, noise=0.2, factor=0.5, random_state=1),
    "линейная граница": (X_linear, y_linear),
}
toy_models = {
    "kNN, k = 15": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "дерево, глубина 5": lambda: DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "бэггинг, 100 деревьев": lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=SEED),
    "случайный лес, 100 деревьев": lambda: RandomForestClassifier(n_estimators=100, random_state=SEED),
}
point_colors = ListedColormap([BLUE, OCHRE])
probability_colors = LinearSegmentedColormap.from_list("класс 0 — класс 1", [BLUE, "white", OCHRE])
boundary_scores = {}

fig, axes = plt.subplots(len(toy_datasets), len(toy_models) + 1, figsize=(19, 11.5))

for row, (data_name, (X_toy, y_toy)) in enumerate(toy_datasets.items()):
    X_toy_train, X_toy_test, y_toy_train, y_toy_test = train_test_split(X_toy, y_toy, test_size=0.4, random_state=SEED)

    x_values = np.linspace(X_toy[:, 0].min() - 0.5, X_toy[:, 0].max() + 0.5, 200)
    y_values = np.linspace(X_toy[:, 1].min() - 0.5, X_toy[:, 1].max() + 0.5, 200)
    grid_x, grid_y = np.meshgrid(x_values, y_values)
    grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

    test_fill = point_colors(y_toy_test)                   # RGBA цвета классов тестовых точек
    test_fill[:, 3] = 0.6                                   # прозрачна только заливка, обводка черная

    data_ax = axes[row, 0]
    data_ax.scatter(X_toy_train[:, 0], X_toy_train[:, 1], c=y_toy_train, cmap=point_colors, s=16, edgecolor="white", lw=0.5)
    data_ax.scatter(X_toy_test[:, 0], X_toy_test[:, 1], facecolors=test_fill, s=16, edgecolors=BLACK, lw=0.8)
    data_ax.set_xlim(x_values[0], x_values[-1])
    data_ax.set_ylim(y_values[0], y_values[-1])
    data_ax.set_ylabel(data_name, fontsize=15)

    for column, (model_name, make_model) in enumerate(toy_models.items(), start=1):
        model = make_model()
        model.fit(X_toy_train, y_toy_train)

        probability = model.predict_proba(grid_points)[:, 1].reshape(grid_x.shape)
        test_accuracy = model.score(X_toy_test, y_toy_test)
        boundary_scores.setdefault(model_name, {})[data_name] = test_accuracy

        ax = axes[row, column]
        extent = (x_values[0], x_values[-1], y_values[0], y_values[-1])
        background = ax.imshow(probability, extent=extent, origin="lower", aspect="auto",
                               cmap=probability_colors, vmin=0, vmax=1, alpha=0.55)
        ax.scatter(X_toy_train[:, 0], X_toy_train[:, 1], c=y_toy_train, cmap=point_colors, s=16, edgecolor="white", lw=0.5)
        ax.scatter(X_toy_test[:, 0], X_toy_test[:, 1], facecolors=test_fill, s=16, edgecolors=BLACK, lw=0.8)
        ax.text(0.97, 0.04, f"{test_accuracy:.2f}", transform=ax.transAxes, ha="right", fontsize=15,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.85))
        if row == 0:
            ax.set_title(model_name, fontsize=15)

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
axes[0, 0].set_title("данные", fontsize=15)

plt.tight_layout()
colorbar = fig.colorbar(background, ax=axes, shrink=0.5, pad=0.01)
colorbar.set_label("предсказанная вероятность класса 1", fontsize=13)
colorbar.ax.tick_params(labelsize=12)
plt.show()

boundary_table = pd.DataFrame(boundary_scores)            # строки — наборы точек, столбцы — модели
show_table(boundary_table, digits=2, axis=1)

Граница дерева состоит из отрезков, параллельных осям. Поэтому одно дерево
хуже всего на кольцах (0.78): круглую границу оно заменяет лестницей. Граница
kNN имеет произвольную форму, не привязанную к осям, и на кольцах kNN точнее
всех (0.87). Бэггинг и лес усредняют много лестниц и делают ступени мельче:
на лунах 0.96 против 0.89 у одного дерева и 0.91 у kNN, на кольцах 0.85 и 0.84.
На наборе с размытой линейной границей классы перекрываются, и все модели
ошибаются чаще (0.74–0.79); граница деревьев остается ступенчатой там, где
истинная граница — прямая. Для деревьев масштабирование признаков
не требуется: вопрос «$x_j \le t$?» не зависит от единиц измерения. kNN сравнивает
расстояния, поэтому перед ним стоит стандартизация `StandardScaler`.

## 10. Изолирующий лес

Случайные деревья применяют не только для классификации и регрессии, но и для
поиска аномалий — объектов, непохожих на остальные. Ответов здесь нет: задача
без учителя. **Изолирующий лес** (isolation forest; Liu, Ting, Zhou, 2008)
опирается на два свойства аномалий: их мало, и они далеко от основной массы.

Объект **изолируют** случайными разбиениями: выбирают случайный признак
и случайный порог между наименьшим и наибольшим значением признака в узле,
оставляют ту часть, где лежит объект, и повторяют, пока он не останется один.
Типичную точку в плотном облаке приходится отделять от многих соседей, и разбиений
нужно много. Аномалию отделяет первое же разбиение, порог которого попал
между ней и облаком, и разбиений нужно мало. Ниже — один такой процесс для самой
типичной точки облака (слева и в увеличении в центре) и для самого далекого
выброса (справа); цвет линии от темного к светлому показывает порядок разбиений.

In [ ]:
rng_iso = np.random.default_rng(0)
cluster_a = rng_iso.normal([0, 0], 0.5, (150, 2))
cluster_b = rng_iso.normal([3, 3], 0.7, (150, 2))
outliers = rng_iso.uniform(-3, 6, (20, 2))

X_iso = np.vstack([cluster_a, cluster_b, outliers])
is_outlier = np.r_[np.zeros(300, dtype=bool), np.ones(20, dtype=bool)]


def isolate(X, point_index, rng):
    """Случайные разбиения до изоляции объекта point_index; возвращает отрезки разбиений."""
    in_node = np.arange(len(X))
    box = [X[:, 0].min(), X[:, 0].max(), X[:, 1].min(), X[:, 1].max()]     # x от, x до, y от, y до
    segments = []

    while len(in_node) > 1:
        feature = rng.integers(2)                                           # случайный признак
        values = X[in_node, feature]
        low, high = values.min(), values.max()
        if low == high:
            break
        threshold = rng.uniform(low, high)                                  # случайный порог

        if feature == 0:
            segments.append(([threshold, threshold], [box[2], box[3]]))
        else:
            segments.append(([box[0], box[1]], [threshold, threshold]))

        point_goes_left = X[point_index, feature] <= threshold
        same_side = (values <= threshold) == point_goes_left
        in_node = in_node[same_side]                                        # остаются объекты с той же стороны

        if point_goes_left:
            box[2 * feature + 1] = threshold
        else:
            box[2 * feature] = threshold
    return segments


distance_to_center = np.linalg.norm(cluster_a, axis=1)
inlier_index = int(np.argmin(distance_to_center))                          # самая типичная точка

distance_to_clusters = np.minimum(np.linalg.norm(outliers, axis=1), np.linalg.norm(outliers - [3, 3], axis=1))
outlier_index = 300 + int(np.argmax(distance_to_clusters))                 # самая далекая от кластеров

def draw_isolation(ax, segments, index, title):
    """Точки, разбиения (цвет от темного к светлому — порядок) и изолируемый объект."""
    step_colors = plt.cm.viridis(np.linspace(0, 0.9, len(segments)))
    ax.scatter(X_iso[:, 0], X_iso[:, 1], s=12, color=GREY)
    for (xs, ys), color in zip(segments, step_colors):
        ax.plot(xs, ys, color=color, lw=1.1)
    ax.scatter(*X_iso[index], s=70, facecolor="none", edgecolor="red", lw=1.8, zorder=3)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("x₁")
    ax.grid(False)


rng_demo = np.random.default_rng(5)
inlier_segments = isolate(X_iso, inlier_index, rng_demo)
outlier_segments = isolate(X_iso, outlier_index, rng_demo)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={"width_ratios": [1, 0.8, 1]})

draw_isolation(axes[0], inlier_segments, inlier_index, f"обычная точка: {len(inlier_segments)} разбиений")
draw_isolation(axes[1], inlier_segments, inlier_index, "то же, увеличение около точки")
draw_isolation(axes[2], outlier_segments, outlier_index, f"аномалия: {len(outlier_segments)} разбиений")

center_x, center_y = X_iso[inlier_index]
axes[1].set_xlim(center_x - 0.4, center_x + 0.4)
axes[1].set_ylim(center_y - 0.4, center_y + 0.4)
zoom_frame = plt.Rectangle((center_x - 0.4, center_y - 0.4), 0.8, 0.8, fill=False, edgecolor=BLACK, lw=1.2)
axes[0].add_patch(zoom_frame)                               # область, показанная в увеличении
for spine in axes[1].spines.values():
    spine.set_visible(True)

axes[0].set_ylabel("x₂")
plt.tight_layout()
plt.show()

Число разбиений до изоляции $h(x)$ случайно. Распределение $h(x)$ для тех же двух
точек по 500 повторам:

In [ ]:
rng_depths = np.random.default_rng(6)
depth_samples = {"обычная точка": [], "аномалия": []}

for _ in range(500):
    inlier_segments = isolate(X_iso, inlier_index, rng_depths)
    outlier_segments = isolate(X_iso, outlier_index, rng_depths)
    depth_samples["обычная точка"].append(len(inlier_segments))
    depth_samples["аномалия"].append(len(outlier_segments))

plt.figure(figsize=(8, 3.8))
bins = np.arange(0, 30) - 0.5
for (name, depths), color in zip(depth_samples.items(), [GREY, OCHRE]):
    plt.hist(depths, bins=bins, color=color, alpha=0.7, label=f"{name}: в среднем {np.mean(depths):.1f}")
plt.xlabel("число разбиений до изоляции h(x)")
plt.ylabel("повторов")
plt.legend()
plt.show()

for name, depths in depth_samples.items():
    print(f"{name}: h(x) в среднем {np.mean(depths):.1f}, медиана {np.median(depths):.0f}")

Типичную точку облака изолируют в среднем за 14.3 разбиения, самый далекий
выброс — за 5.9. Длина пути и служит мерой аномальности: чем она короче,
тем объект необычнее.

**Алгоритм.** Изолирующий лес строит $B$ таких деревьев (по умолчанию 100), каждое
на случайной подвыборке из $\psi$ объектов без возвращения (`max_samples`, по умолчанию
$\psi = \min(256, n)$). Для объекта $x$ вычисляют среднюю по деревьям длину пути от корня
до листа $\mathbb{E}\,h(x)$ и сравнивают ее с типичной длиной пути в дереве на $\psi$ объектах:

$$c(\psi) = 2H(\psi - 1) - \frac{2(\psi - 1)}{\psi}, \qquad H(i) \approx \ln i + 0.5772,$$

где $H(i)$ — гармоническое число; $c(\psi)$ — средняя длина пути при неудачном поиске
в двоичном дереве поиска на $\psi$ объектах. **Оценка аномальности**

$$s(x) = 2^{-\mathbb{E}\,h(x) / c(\psi)}$$

близка к 1, если объект изолируют намного быстрее типичного ($\mathbb{E}\,h(x) \ll c(\psi)$),
равна 0.5 при $\mathbb{E}\,h(x) = c(\psi)$ и меньше 0.5 при более длинном пути. Подвыборка
из 256 объектов нужна, чтобы плотные скопления аномалий не маскировали друг
друга, и делает метод быстрым. В sklearn — `IsolationForest`: деревья растят
до высоты $\lceil \log_2 \psi \rceil$ (8 при $\psi = 256$), а к длине пути до листа, в котором осталось
$n_{\text{лист}}$ объектов, прибавляют $c(n_{\text{лист}})$ — ожидаемую длину оставшегося пути.
Метод `score_samples` возвращает $-s(x)$, параметр `contamination` задает ожидаемую
долю аномалий и порог для `predict`.

Качество поиска аномалий оценивают по добавленным выбросам метрикой ROC AUC
(receiver operating characteristic, area under the curve, площадь под ROC-кривой):
это вероятность того, что случайный выброс получит $s(x)$ больше, чем случайная
обычная точка.

In [ ]:
iso_forest = IsolationForest(n_estimators=200, max_samples=256, random_state=SEED)
iso_forest.fit(X_iso)

anomaly_score = -iso_forest.score_samples(X_iso)          # s(x): sklearn возвращает -s(x)
auc = roc_auc_score(is_outlier, anomaly_score)

x_values = np.linspace(-3.5, 6.5, 250)
y_values = np.linspace(-3.5, 6.5, 250)
grid_x, grid_y = np.meshgrid(x_values, y_values)
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]
grid_score = -iso_forest.score_samples(grid_points).reshape(grid_x.shape)

top_20 = np.argsort(anomaly_score)[-20:]                  # 20 объектов с наибольшей оценкой

fig, ax = plt.subplots(figsize=(9, 7))
filled = ax.contourf(grid_x, grid_y, grid_score, levels=20, cmap="viridis")
ax.scatter(X_iso[~is_outlier, 0], X_iso[~is_outlier, 1], s=12, color="white", edgecolor=BLACK, lw=0.3, label="обычные точки")
ax.scatter(X_iso[is_outlier, 0], X_iso[is_outlier, 1], s=40, color=OCHRE, edgecolor=BLACK, label="добавленные выбросы")
ax.scatter(X_iso[top_20, 0], X_iso[top_20, 1], s=160, facecolor="none", edgecolor="red", lw=1.5, label="20 наибольших s(x)")
fig.colorbar(filled, ax=ax, label="оценка аномальности s(x)")
ax.set_title(f"изолирующий лес: ROC AUC по добавленным выбросам {auc:.3f}")
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")
ax.legend(loc="lower left")
ax.grid(False)
plt.show()

found = is_outlier[top_20].sum()
print(f"ROC AUC по добавленным выбросам: {auc:.3f}")
print(f"среди 20 объектов с наибольшей s(x) добавленных выбросов: {found}")
print(f"s(x): обычные точки в среднем {anomaly_score[~is_outlier].mean():.3f}, выбросы {anomaly_score[is_outlier].mean():.3f}")

Оценка $s(x)$ высока вдали от облаков и мала внутри них. По добавленным
выбросам ROC AUC 0.947; среди 20 объектов с наибольшей $s(x)$ 15 — выбросы.
Остальные выбросы случайно попали внутрь или рядом с облаками и по положению
не отличаются от обычных точек. Средняя $s(x)$ обычных точек 0.445, выбросов 0.625.

## Итоги

- дерево делит пространство признаков на прямоугольные области и в каждой
  предсказывает константу: самый частый класс или среднее ответов;
- вопрос в узле выбирают жадно, перебором признаков и порогов,
  по наибольшему уменьшению индекса Джини, энтропии или MSE;
- без ограничений дерево переобучается; рост ограничивают `max_depth`,
  `min_samples_leaf`, `max_leaf_nodes` и подбирают их кросс-валидацией;
- MDI — доля порядка, наведенного признаком на обучающей выборке;
  перестановочная важность — падение качества на тестовом сете после
  перемешивания признака; она не зависит от переобучения на обучающей выборке,
  но занижает важность коррелированных признаков;
- пропуски дерево обрабатывает само: встроенным правилом sklearn, спуском
  в оба поддерева или заполнением константой вне диапазона;
- при фиксированных областях дерево — линейная модель на индикаторах листьев;
- граница дерева ступенчатая, вдоль осей; граница kNN произвольной формы;
- глубокое дерево имеет малое смещение и большой разброс;
- бэггинг усредняет модели, обученные на бутстрэп-выборках: смещение
  такое же, как у одной модели, разброс уменьшается до $\rho\sigma^2$;
- оценка вне выборки (OOB) — качество предсказаний каждого объекта деревьями,
  которые его не видели; получается без отдельного тестового сета;
- случайный лес — бэггинг плюс метод случайных подпространств в каждом узле:
  деревья становятся менее похожими, $\rho$ и предел разброса $\rho\sigma^2$ уменьшаются,
  а смещение растет, если `max_features` слишком мало;
- изолирующий лес ищет аномалии без ответов: аномалию отделяют меньше
  случайных разбиений, оценка $s(x) = 2^{-\mathbb{E}h(x)/c(\psi)}$ близка к 1 у аномалий.

Следующее занятие — градиентный бустинг (gradient boosting): деревья строят
последовательно, и каждое исправляет ошибки предыдущих.